<a href="https://colab.research.google.com/github/brianvngan/SyntheticUltrasoundSpleen/blob/main/SpleenCycleGan.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

### Discriminator Model (`discriminator_model.py`)

This script defines the `Discriminator` model, a crucial component of the CycleGAN architecture. The discriminator's role is to distinguish between real images and fake images generated by the generator.

*   **`Block` Class**: This is a helper class representing a basic building block used within the discriminator. It consists of a `Conv2d` layer, `InstanceNorm2d` for normalization, and a `LeakyReLU` activation function. The `padding_mode='reflect'` is used to handle image borders more naturally.

*   **`Discriminator` Class**: This is the main discriminator model.
    *   **`__init__`**:
        *   The `initial` block is the first layer, which is a `Conv2d` followed by `LeakyReLU`. It takes the input image and reduces its spatial dimensions.
        *   It then constructs a series of `Block`s. These blocks progressively downsample the image (stride=2) until the last block, which uses a stride of 1. This creates a patch-based discriminator, meaning it classifies patches of the image as real or fake, rather than the entire image.
        *   The final `Conv2d` layer maps the features to a single output channel, representing the discriminator's confidence score for each patch.
    *   **`forward`**: The input `x` first passes through the `initial` layer, then through the sequence of `Block`s. Finally, a `sigmoid` activation is applied to the output of the last convolutional layer. The sigmoid output is a probability score (between 0 and 1) indicating how 'real' each patch of the input image is perceived to be by the discriminator.

### Discriminator Test Function

This code block tests the `Discriminator` model's forward pass by creating a dummy input tensor and checking the output shape.

*   **`x = torch.randn((5, 3, 256, 256))`**: Creates a random tensor `x` representing a batch of 5 images, each with 3 color channels (RGB) and a resolution of 256x256 pixels. This simulates input images for the discriminator.
*   **`model = Discriminator(in_channels=3)`**: Instantiates the `Discriminator` model, expecting 3 input channels.
*   **`preds = model(x)`**: Passes the dummy input `x` through the `Discriminator` model to get the predictions.
*   **`print(preds.shape)`**: Prints the shape of the output predictions. For a 256x256 input, a typical patch discriminator structure will produce a smaller spatial output (e.g., 30x30), with one channel per patch indicating the real/fake probability.

### Generator Model (`generator_model.py`)

This script defines the `Generator` model, responsible for transforming images from one domain to another (e.g., horse to zebra). It uses a U-Net-like architecture with residual blocks.

*   **`ConvBlock` Class**: A versatile convolutional block. It can perform either downsampling (`Conv2d`) or upsampling (`ConvTranspose2d`). It includes `InstanceNorm2d` and an optional `ReLU` activation.

*   **`ResidualBlock` Class**: Implements a standard residual connection. It consists of two `ConvBlock`s with the same number of channels, and its output is added to its input, helping with gradient flow and training deeper networks.

*   **`Generator` Class**: This is the main generator model.
    *   **`__init__`**:
        *   **`initial`**: A convolutional layer that processes the input image initially.
        *   **`down_blocks`**: A list of `ConvBlock`s that perform downsampling (reducing spatial dimensions and increasing feature channels), akin to an encoder.
        *   **`residual_blocks`**: A sequence of `ResidualBlock`s applied at the bottleneck of the U-Net-like structure. These blocks help capture complex mappings without losing information.
        *   **`up_blocks`**: A list of `ConvBlock`s with `down=False` (meaning they are `ConvTranspose2d`), performing upsampling (increasing spatial dimensions and decreasing feature channels), akin to a decoder.
        *   **`last`**: The final convolutional layer that maps the feature channels back to the original image channels.
    *   **`forward`**: The input `x` flows sequentially through the `initial` layer, `down_blocks`, `residual_blocks`, and `up_blocks`. The final `last` layer applies a `tanh` activation to ensure the output pixel values are in the range of -1 to 1, consistent with the normalized input image range.

### Generator Test Function

This code block tests the `Generator` model's forward pass to ensure it can process input images and produce an output of the expected shape.

*   **`img_channels = 3`**: Sets the number of input and output image channels to 3 (for RGB images).
*   **`img_size = 256`**: Sets the image resolution to 256x256 pixels.
*   **`X = torch.randn((2, img_channels, img_size, img_size))`**: Creates a random tensor `X` representing a batch of 2 images, each with 3 color channels and a 256x256 resolution. This simulates input images for the generator.
*   **`gen = Generator(img_channels, 9)`**: Instantiates the `Generator` model, expecting 3 input/output channels and using 9 residual blocks (a common configuration for CycleGAN).
*   **`print(gen(X).shape)`**: Passes the dummy input `X` through the `Generator` model and prints the shape of the output. The expected output shape is `[2, 3, 256, 256]`, confirming that the generator produces images with the same dimensions as the input.

### Configuration File (`config.py`)

This file centralizes all the hyperparameters and settings for the CycleGAN training process. This makes it easy to modify experimental parameters without changing the core training logic.

*   **`DEVICE`**: Automatically detects if a CUDA-enabled GPU is available and sets the device to 'cuda', otherwise 'cpu'.
*   **`TRAIN_DIR`, `VAL_DIR`**: Paths to the training and validation datasets. Initially, these point to the same directory, which might be adjusted later.
*   **`BATCH_SIZE`**: Number of images processed in one training iteration.
*   **`LEARNING_RATE`**: The step size for the optimizer during training.
*   **`LAMBDA_IDENTITY`, `LAMBDA_CYCLE`**: Weights for the identity and cycle consistency losses, respectively. `LAMBDA_CYCLE` is usually high to enforce consistency, while `LAMBDA_IDENTITY` can be 0 or small.
*   **`NUM_WORKERS`**: Number of subprocesses to use for data loading, improving loading speed.
*   **`NUM_EPOCHS`**: Total number of times the entire dataset will be passed through the network.
*   **`LOAD_MODEL`, `SAVE_MODEL`**: Boolean flags to control whether to load a pre-trained model or save the current model during training.
*   **`CHECKPOINT_GEN_H`, `CHECKPOINT_GEN_Z`, `CHECKPOINT_CRITIC_H`, `CHECKPOINT_CRITIC_Z`**: Filenames for saving the generator and discriminator model checkpoints for both horse-to-zebra and zebra-to-horse translations.
*   **`transforms`**: Defines image augmentation and preprocessing steps using the `albumentations` library:
    *   `A.Resize`: Resizes images to 256x256 pixels.
    *   `A.HorizontalFlip`: Randomly flips images horizontally (p=0.5).
    *   `A.Normalize`: Normalizes pixel values to a range of -1 to 1 (mean=0.5, std=0.5 for each channel), which is standard for `tanh` output in GANs.
    *   `ToTensorV2`: Converts images to PyTorch tensors.
    *   `additional_targets`: Ensures that transformations are applied consistently to both image domains (e.g., if a horse image is flipped, the corresponding zebra image should also be considered flipped if it were to undergo transformation in the same way, though for CycleGAN, `image0` just ensures consistent transforms across different image types if needed).

### Config Import and Display

This block imports the `config.py` file and prints some of its key variables to verify that the configuration is loaded correctly. This is a quick way to check the active settings before starting the training process.

*   **`import config`**: Imports the `config` module.
*   **`print(f"Device: {config.DEVICE}")`**: Prints the detected device (GPU or CPU).
*   **`print(f"Learning Rate: {config.LEARNING_RATE}")`**: Prints the learning rate.
*   **`print(f"Train Directory: {config.TRAIN_DIR}")`**: Prints the training data directory.
*   **`print(f"Lambda Cycle: {config.LAMBDA_CYCLE}")`**: Prints the weight for the cycle consistency loss.

### Utilities File (`utils.py`)

This script provides utility functions for saving and loading model checkpoints and for ensuring reproducibility by setting random seeds.

*   **`save_checkpoint(model, optimizer, filename)`**: Saves the current state of a PyTorch model and its optimizer. This includes the model's `state_dict` (learnable parameters) and the optimizer's `state_dict`. This allows training to be resumed from a specific point.

*   **`load_checkpoint(checkpoint_file, model, optimizer, lr)`**: Loads a previously saved checkpoint into a model and optimizer. It also explicitly sets the learning rate of the loaded optimizer, which is important because the optimizer's state_dict might retain the learning rate from the time it was saved.

*   **`seed_everything(seed=42)`**: Sets random seeds for Python's `random` module, `numpy`, and `torch` (including CUDA). This is crucial for reproducibility, ensuring that running the same code multiple times yields the same results, which is very helpful for debugging and comparing experiments. `torch.backends.cudnn.deterministic = True` and `torch.backends.cudnn.benchmark = False` further help ensure deterministic behavior, especially with CUDA operations.

### Install `opendatasets`

This command installs the `opendatasets` Python library. This library simplifies downloading datasets from Kaggle directly into the Colab environment, handling authentication and download processes.

### Download CycleGAN Dataset

This code block uses the `opendatasets` library to download the 'CycleGAN' dataset from Kaggle. Specifically, it targets the 'horse2zebra' portion of the dataset, which is commonly used for demonstrating CycleGAN's capabilities.

*   **`import opendatasets as od`**: Imports the `opendatasets` library.
*   **`od.download('https://www.kaggle.com/datasets/suyashdamle/cyclegan?select=horse2zebra')`**: This function initiates the download. When executed, it prompts the user to enter their Kaggle username and API key (which can be obtained from Kaggle account settings). The `?select=horse2zebra` parameter ensures only the horse2zebra subset of the larger CycleGAN dataset is downloaded.

### Dataset Loading (`dataset.py`)

This script defines a custom PyTorch `Dataset` class, `HorseZebraDataset`, to efficiently load and preprocess the horse and zebra images for CycleGAN training.

*   **`HorseZebraDataset` Class**:
    *   **`__init__`**:
        *   Takes `root_zebra` and `root_horse` as paths to the directories containing zebra and horse images, respectively.
        *   `transform`: An optional argument to apply image transformations (defined in `config.py`).
        *   `zebra_images` and `horse_images`: Lists all image filenames in their respective directories.
        *   `length_dataset`: Determines the effective length of the dataset for iteration. It's set to the maximum of the two image list lengths to ensure all images from both domains are potentially seen during an epoch, even if one domain has fewer images.
    *   **`__len__`**: Returns the `length_dataset`, allowing the DataLoader to know how many items to iterate over.
    *   **`__getitem__`**: This is the core method for loading individual data samples.
        *   It uses the `index` (modulo the length of each list) to retrieve corresponding zebra and horse image filenames. This ensures that if one domain has fewer images, they are recycled, preventing `IndexError`.
        *   `Image.open(...).convert("RGB")`: Opens the image files using PIL and converts them to RGB format.
        *   `np.array(...)`: Converts the PIL image to a NumPy array.
        *   **`if self.transform:`**: If transformations are defined, they are applied to both the zebra and horse images. The `additional_targets` in the `config.transforms` are crucial here to ensure both images are augmented consistently (e.g., if a horizontal flip occurs, it applies to both).
        *   Returns the transformed zebra and horse images as PyTorch tensors.

### Training Function (`train.py`)

This script contains the main training loop for the CycleGAN model, including the `train_fn` (training step for one epoch) and the `main` function (overall training orchestration).

*   **`train_fn(...)`**: This function encapsulates a single training step for one epoch.
    *   **Discriminator Training**:
        *   `fake_horse = gen_H(zebra)`: Generates a fake horse image from a real zebra image.
        *   `D_H_real = disc_H(horse)`: Discriminator H predicts on real horse images.
        *   `D_H_fake = disc_H(fake_horse.detach())`: Discriminator H predicts on fake horse images (detached to prevent gradients from flowing back to the generator).
        *   Calculates BCEWithLogitsLoss (or MSE in this implementation) for real images (target 1s) and fake images (target 0s) for both Discriminator H and Discriminator Z.
        *   `D_loss = (D_H_loss + D_Z_loss) / 2`: Combines discriminator losses.
        *   Updates Discriminator optimizers using mixed precision (`d_scaler`).
    *   **Generator Training**:
        *   **Adversarial Loss**: `loss_G_H = mse(D_H_fake, torch.ones_like(D_H_fake))` and `loss_G_Z = mse(D_Z_fake, torch.ones_like(D_Z_fake))`. The generators try to fool their respective discriminators by making fake images look real (target 1s).
        *   **Cycle Consistency Loss**:
            *   `cycle_zebra = gen_Z(fake_horse)`: Generates a zebra from the fake horse.
            *   `cycle_horse = gen_H(fake_zebra)`: Generates a horse from the fake zebra.
            *   `cycle_zebra_loss = l1(zebra, cycle_zebra)`: Measures how close the cycled zebra image is to the original real zebra image.
            *   `cycle_horse_loss = l1(horse, cycle_horse)`: Similar for horse images. These losses (`LAMBDA_CYCLE`) ensure that the translation is reversible.
        *   **Identity Loss (Optional)**:
            *   `identity_zebra = gen_Z(zebra)`: Generator Z tries to generate a zebra from a real zebra. If it's a good generator, it should produce something close to the original zebra.
            *   `identity_horse = gen_H(horse)`: Similar for Generator H. These losses (`LAMBDA_IDENTITY`) encourage the generators to preserve color composition.
        *   `G_loss`: Combines all generator losses.
        *   Updates Generator optimizers using mixed precision (`g_scaler`).
    *   **Image Saving**: Every 200 iterations, it saves generated fake horse and zebra images to the `saved_images` directory, allowing visual inspection of training progress. The images are unnormalized (`* 0.5 + 0.5`) before saving.
    *   **Progress Bar**: Updates the `tqdm` progress bar with discriminator loss metrics.

*   **`main()` Function**:
    *   **Model Initialization**: Instantiates four models: two discriminators (`disc_H`, `disc_Z`) and two generators (`gen_Z`, `gen_H`) for the two mapping directions.
    *   **Optimizer Initialization**: Sets up `Adam` optimizers for both discriminators and generators, using parameters defined in `config.py`.
    *   **Loss Functions**: Defines `L1Loss` for cycle and identity consistency and `MSELoss` for adversarial loss.
    *   **Checkpoint Loading**: If `config.LOAD_MODEL` is `True`, it loads pre-trained model weights and optimizer states.
    *   **Dataset and DataLoader**: Creates `HorseZebraDataset` instances for both training and validation (though validation is not explicitly used in the `train_fn` loop in this simplified example) and wraps them in `DataLoader`s for batching and shuffling.
    *   **Mixed Precision**: Initializes `torch.cuda.amp.GradScaler` for mixed-precision training, which helps speed up training and reduce memory usage on compatible GPUs.
    *   **Training Loop**: Iterates for `config.NUM_EPOCHS` times, calling `train_fn` for each epoch.
    *   **Checkpoint Saving**: If `config.SAVE_MODEL` is `True`, it saves the models after each epoch.

### Training Execution

This block imports and reloads the necessary modules (`config`, `dataset`, `train`) to ensure that any recent changes to these files are applied. Then, it calls the `main()` function from `train.py` to start the CycleGAN training process.

*   **`importlib.reload(sys.modules['module_name'])`**: This pattern is useful in interactive environments like Colab to reload modules that might have been modified and re-written using `%%writefile` without restarting the kernel.
*   **`import train`**: Imports the `train` module.
*   **`train.main()`**: Executes the main training function, which orchestrates the entire training loop for the CycleGAN model as defined in `train.py`.

### Inspecting the Downloaded Dataset Structure

This command lists the contents of the `cyclegan` directory recursively (`-R`) to show the structure of the downloaded dataset. This helps verify that the data has been downloaded correctly and that the `horse2zebra` subdirectories (`trainA`, `trainB`, `testA`, `testB`) are present as expected by the `dataset.py` and `config.py` files.

*   **`!ls -R cyclegan`**: The `!` prefix allows running shell commands directly in a Colab code cell. `ls -R` lists directory contents recursively.

#Discriminator


In [ ]:
%%writefile discriminator_model.py
import torch
import torch.nn as nn


class Block(nn.Module):
  def __init__(self, in_channels, out_channels, stride):
    super().__init__()
    self.conv = nn.Sequential(
        # Slides a small filter (kernel) over an image to extract features like edges, textures, and patterns
        nn.Conv2d(in_channels, out_channels, 4, stride, 1, bias=True, padding_mode="reflect"),
        nn.InstanceNorm2d(out_channels), #normalizes activations
        nn.LeakyReLU(0.2), #introduce non-linearity
    )

  def forward(self, x):
    return self.conv(x)

class Discriminator(nn.Module):
  def __init__(self, in_channels=1, features=[64, 128, 256, 512]): # Changed in_channels to 1 for grayscale
    super().__init__()
    self.initial = nn.Sequential(
        nn.Conv2d(
            in_channels,
            features[0],
            kernel_size=4,
            stride=2,
            padding=1,
            padding_mode="reflect",
        ),
        nn.LeakyReLU(0.2),
    )

    layers = []
    in_channels = features[0]
    for feature in features[1:]:
      layers.append(Block(in_channels, feature, stride=1 if feature==features[-1] else 2))
      in_channels = feature
    layers.append(nn.Conv2d(in_channels, 1, kernel_size=4, stride=1, padding=1, padding_mode="reflect"))
    self.model = nn.Sequential(*layers)
  def forward(self, x):
    x = self.initial(x)
    return torch.sigmoid(self.model(x))

Overwriting discriminator_model.py


In [ ]:
import torch
from discriminator_model import Discriminator

def test():
  x = torch.randn((5, 3, 256, 256))
  model = Discriminator(in_channels=3)
  preds = model(x)
  print(preds.shape)

if __name__ == "__main__":
  test()

torch.Size([5, 1, 30, 30])


#Generator


In [ ]:
%%writefile generator_model.py
import torch
import torch.nn as nn

class ConvBlock(nn.Module):
  def __init__(self, in_channels, out_channels, down=True, use_act=True, **kwargs):
    super().__init__()
    self.conv = nn.Sequential(
        nn.Conv2d(in_channels, out_channels, padding_mode="reflect", **kwargs)
        if down
        else nn.ConvTranspose2d(in_channels, out_channels, **kwargs),
        nn.InstanceNorm2d(out_channels),
        nn.ReLU(inplace=True) if use_act else nn.Identity()
    )

  def forward(self, x):
    return self.conv(x)

class ResidualBlock(nn.Module):
  def __init__(self, channels):
    super().__init__()
    self.block = nn.Sequential(
        ConvBlock(channels, channels, kernel_size=3, padding=1),
        ConvBlock(channels, channels, use_act=False, kernel_size=3, padding=1),
    )
  def forward(self, x):
    return x + self.block(x)

class Generator(nn.Module):
  def __init__(self, img_channels=1, num_features = 64, num_residuals=9): # Changed img_channels to 1 for grayscale
    super().__init__()
    self.initial = nn.Sequential(
        nn.Conv2d(img_channels, num_features, kernel_size=7, stride=1, padding=3, padding_mode="reflect"),
        nn.ReLU(inplace=True),
    )
    self.down_blocks = nn.ModuleList(
        [
            ConvBlock(num_features, num_features*2, kernel_size=3, stride=2, padding=1),
            ConvBlock(num_features*2, num_features*4, kernel_size=3, stride=2, padding=1),
        ]
    )
    self.residual_blocks = nn.Sequential(
        *[ResidualBlock(num_features*4) for _ in range(num_residuals)]
    )
    self.up_blocks = nn.ModuleList(
        [
            ConvBlock(num_features*4, num_features*2, down=False, kernel_size=3, stride=2, padding=1, output_padding=1),
            ConvBlock(num_features*2, num_features*1, down=False, kernel_size=3, stride=2, padding=1, output_padding=1),
        ]
    )

    self.last = nn.Conv2d(num_features*1, img_channels, kernel_size=7, stride=1, padding=3, padding_mode="reflect")

  def forward(self, x):
    x = self.initial(x)
    for layer in self.down_blocks:
      x = layer(x)
    x = self.residual_blocks(x)
    for layer in self.up_blocks:
      x = layer(x)
    return torch.tanh(self.last(x))

Overwriting generator_model.py


In [ ]:
from generator_model import Generator

def test():
  img_channels = 3
  img_size = 256
  X = torch.randn((2, img_channels, img_size, img_size))
  gen = Generator(img_channels, 9)
  print(gen(X).shape)

if __name__ == "__main__":
  test()

torch.Size([2, 3, 256, 256])


##Configs
First, let's create a `config.py` file. We can write its content directly to the file system using the `%%writefile` magic command.

In [ ]:
%%writefile config.py

import torch
import albumentations as A
from albumentations.pytorch import ToTensorV2

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
TRAIN_DIR = "/content/drive/MyDrive/Processed Datasets/US.v7i.yolov8-obb Roboflow/train/images"
VAL_DIR = "/content/drive/MyDrive/Processed Datasets/US.v7i.yolov8-obb Roboflow/test/images"
BATCH_SIZE = 1
LEARNING_RATE = 1e-5
LAMBDA_IDENTITY = 0.0 # Consider tuning this if needed, e.g., 0.1 * LAMBDA_CYCLE
LAMBDA_CYCLE = 10
NUM_WORKERS = 2 # Reduced to optimize DataLoader performance
NUM_EPOCHS = 10
LOAD_MODEL = False # Set to False for new training
SAVE_MODEL = True
CHECKPOINT_GEN_A_TO_B = "gen_A_to_B.pth.tar" # Generator from Domain A to B
CHECKPOINT_GEN_B_TO_A = "gen_B_to_A.pth.tar" # Generator from Domain B to A
CHECKPOINT_CRITIC_A = "critic_A.pth.tar"     # Discriminator for Domain A
CHECKPOINT_CRITIC_B = "critic_B.pth.tar"     # Discriminator for Domain B

transforms = A.Compose(
    [
        A.Resize(width=256, height=256),
        A.HorizontalFlip(p=0.5),
        A.Normalize(mean=[0.5], std=[0.5], max_pixel_value=255), # Adjusted for single-channel grayscale
        ToTensorV2(),
    ],
    additional_targets={"image0": "image"},
)

Overwriting config.py


In [ ]:
import config

print(f"Device: {config.DEVICE}")
print(f"Learning Rate: {config.LEARNING_RATE}")
print(f"Train Directory: {config.TRAIN_DIR}")
print(f"Lambda Cycle: {config.LAMBDA_CYCLE}")

Device: cuda
Learning Rate: 1e-05
Train Directory: /content/drive/MyDrive/Processed Datasets/US.v7i.yolov8-obb Roboflow/train/images
Lambda Cycle: 10


##Utils


In [ ]:
%%writefile utils.py
import random, torch, os, numpy as np
import torch.nn as nn
import config
import copy

def save_checkpoint(model, optimizer, filename="my_checkpoint.pth.tar"):
    print("=> Saving checkpoint")
    checkpoint = {
        "state_dict": model.state_dict(),
        "optimizer": optimizer.state_dict(),
    }
    torch.save(checkpoint, filename)


def load_checkpoint(checkpoint_file, model, optimizer, lr):
    print("=> Loading checkpoint")
    checkpoint = torch.load(checkpoint_file, map_location=config.DEVICE)
    model.load_state_dict(checkpoint["state_dict"])
    optimizer.load_state_dict(checkpoint["optimizer"])

    # If we don't do this then it will just have learning rate of old checkpoint
    # and it will lead to many hours of debugging \:
    for param_group in optimizer.param_groups:
        param_group["lr"] = lr


def seed_everything(seed=42):
    os.environ["PYTHONHASHSEED"] = str(seed)
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False

Overwriting utils.py


#Dataset Loading


In [ ]:
!pip install opendatasets

In [ ]:
#import opendatasets as od
#od.download('https://www.kaggle.com/datasets/ignaciorlando/ussimandsegm')

In [ ]:
import importlib
import sys

# Reload config module to pick up new paths
if 'config' in sys.modules:
    importlib.reload(sys.modules['config'])
import config

print(f"Device: {config.DEVICE}")
print(f"Learning Rate: {config.LEARNING_RATE}")
print(f"Train Directory: {config.TRAIN_DIR}")
print(f"Validation Directory: {config.VAL_DIR}")
print(f"Lambda Cycle: {config.LAMBDA_CYCLE}")

Device: cuda
Learning Rate: 1e-05
Train Directory: /content/drive/MyDrive/Processed Datasets/US.v7i.yolov8-obb Roboflow/train/images
Validation Directory: /content/drive/MyDrive/Processed Datasets/US.v7i.yolov8-obb Roboflow/test/images
Lambda Cycle: 10


In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [ ]:
%%writefile dataset.py
import torch
from PIL import Image
import os
from torch.utils.data import Dataset
import numpy as np

class SpleenDataset(Dataset):
    def __init__(self, root_domain_A, root_domain_B, transform=None):
        self.root_domain_A = root_domain_A
        self.root_domain_B = root_domain_B
        self.transform = transform

        self.domain_A_images = os.listdir(root_domain_A)
        self.domain_B_images = os.listdir(root_domain_B)
        self.length_dataset = max(len(self.domain_A_images), len(self.domain_B_images)) # 1000, 1500
        self.domain_A_len = len(self.domain_A_images)
        self.domain_B_len = len(self.domain_B_images)

    def __len__(self):
        return self.length_dataset

    def __getitem__(self, index):
        domain_A_img_name = self.domain_A_images[index % self.domain_A_len]
        domain_B_img_name = self.domain_B_images[index % self.domain_B_len]

        domain_A_path = os.path.join(self.root_domain_A, domain_A_img_name)
        domain_B_path = os.path.join(self.root_domain_B, domain_B_img_name)

        # Changed to convert("L") for grayscale images
        domain_A_img = np.array(Image.open(domain_A_path).convert("L"))
        domain_B_img = np.array(Image.open(domain_B_path).convert("L"))

        if self.transform:
            # For grayscale, need to ensure the transformation expects 1 channel
            augmentations = self.transform(image=domain_A_img, image0=domain_B_img)
            domain_A_img = augmentations["image"]
            domain_B_img = augmentations["image0"]

        return domain_A_img, domain_B_img

Overwriting dataset.py


Train Function

In [ ]:
%%writefile train.py
import torch
from dataset import SpleenDataset # Updated dataset class
import sys
from utils import save_checkpoint, load_checkpoint
from torch.utils.data import DataLoader
import torch.nn as nn
import torch.optim as optim
import config
from tqdm import tqdm
from torchvision.utils import save_image
from discriminator_model import Discriminator
from generator_model import Generator
import os

def train_fn(
    disc_A, disc_B, gen_B_to_A, gen_A_to_B, loader, opt_disc, opt_gen, l1, mse, d_scaler, g_scaler
):
    B_reals = 0 # Track real predictions for Domain B
    B_fakes = 0 # Track fake predictions for Domain B
    loop = tqdm(loader, leave=True)

    for idx, (domain_A, domain_B) in enumerate(loop): # Renamed zebra to domain_A, horse to domain_B
        domain_A = domain_A.to(config.DEVICE)
        domain_B = domain_B.to(config.DEVICE)

        # Train Discriminators for Domain A and Domain B
        with torch.amp.autocast(device_type=config.DEVICE.split(':')[0]): # Updated to torch.amp.autocast
            fake_domain_B = gen_A_to_B(domain_A) # Gen A->B generates fake Domain B (from A)
            D_B_real = disc_B(domain_B)
            D_B_fake = disc_B(fake_domain_B.detach())
            B_reals += D_B_real.mean().item()
            B_fakes += D_B_fake.mean().item()
            D_B_real_loss = mse(D_B_real, torch.ones_like(D_B_real))
            D_B_fake_loss = mse(D_B_fake, torch.zeros_like(D_B_fake))
            D_B_loss = D_B_real_loss + D_B_fake_loss

            fake_domain_A = gen_B_to_A(domain_B) # Gen B->A generates fake Domain A (from B)
            D_A_real = disc_A(domain_A)
            D_A_fake = disc_A(fake_domain_A.detach())
            D_A_real_loss = mse(D_A_real, torch.ones_like(D_A_real))
            D_A_fake_loss = mse(D_A_fake, torch.zeros_like(D_A_fake))
            D_A_loss = D_A_real_loss + D_A_fake_loss

            # put it togethor
            D_loss = (D_B_loss + D_A_loss) / 2

        opt_disc.zero_grad()
        d_scaler.scale(D_loss).backward()
        d_scaler.step(opt_disc)
        d_scaler.update()

        # Train Generators A->B and B->A
        with torch.amp.autocast(device_type=config.DEVICE.split(':')[0]): # Updated to torch.amp.autocast
            # adversarial loss for both generators
            D_B_fake = disc_B(fake_domain_B)
            D_A_fake = disc_A(fake_domain_A)
            loss_G_B_to_A = mse(D_B_fake, torch.ones_like(D_B_fake))
            loss_G_A_to_B = mse(D_A_fake, torch.ones_like(D_A_fake))

            # cycle loss
            cycle_domain_A = gen_B_to_A(fake_domain_B) # A -> B -> A
            cycle_domain_B = gen_A_to_B(fake_domain_A) # B -> A -> B
            cycle_domain_A_loss = l1(domain_A, cycle_domain_A)
            cycle_domain_B_loss = l1(domain_B, cycle_domain_B)

            # identity loss (remove these for efficiency if you set lambda_identity=0)
            identity_domain_A = gen_B_to_A(domain_A) # B->A generator on A domain image, should be A
            identity_domain_B = gen_A_to_B(domain_B) # A->B generator on B domain image, should be B
            identity_domain_A_loss = l1(domain_A, identity_domain_A)
            identity_domain_B_loss = l1(domain_B, identity_domain_B)

            # add all togethor
            G_loss = (
                loss_G_A_to_B # Generator A->B tries to fool Disc B
                + loss_G_B_to_A # Generator B->A tries to fool Disc A
                + cycle_domain_A_loss * config.LAMBDA_CYCLE
                + cycle_domain_B_loss * config.LAMBDA_CYCLE
                + identity_domain_B_loss * config.LAMBDA_IDENTITY # Identity loss for A->B generator
                + identity_domain_A_loss * config.LAMBDA_IDENTITY # Identity loss for B->A generator
            )

        opt_gen.zero_grad()
        g_scaler.scale(G_loss).backward()
        g_scaler.step(opt_gen)
        g_scaler.update()

        if idx % 200 == 0:
            os.makedirs('saved_images', exist_ok=True)
            save_image(fake_domain_B * 0.5 + 0.5, f"saved_images/domain_B_{idx}.png")
            save_image(fake_domain_A * 0.5 + 0.5, f"saved_images/domain_A_{idx}.png")

        loop.set_postfix(B_real=B_reals / (idx + 1), B_fake=B_fakes / (idx + 1))


def main():
    # Renamed discriminators and generators
    disc_A = Discriminator(in_channels=1).to(config.DEVICE) # Set to 1 channel for grayscale
    disc_B = Discriminator(in_channels=1).to(config.DEVICE) # Set to 1 channel for grayscale
    gen_B_to_A = Generator(img_channels=1, num_residuals=9).to(config.DEVICE) # Maps B to A (1 channel)
    gen_A_to_B = Generator(img_channels=1, num_residuals=9).to(config.DEVICE) # Maps A to B (1 channel)
    opt_disc = optim.Adam(
        list(disc_A.parameters()) + list(disc_B.parameters()),
        lr=config.LEARNING_RATE,
        betas=(0.5, 0.999),
    )

    opt_gen = optim.Adam(
        list(gen_B_to_A.parameters()) + list(gen_A_to_B.parameters()), # Updated generators
        lr=config.LEARNING_RATE,
        betas=(0.5, 0.999),
    )

    L1 = nn.L1Loss()
    mse = nn.MSELoss()

    if config.LOAD_MODEL:
        load_checkpoint(
            config.CHECKPOINT_GEN_A_TO_B,
            gen_A_to_B, # Load into gen_A_to_B
            opt_gen,
            config.LEARNING_RATE,
        )
        load_checkpoint(
            config.CHECKPOINT_GEN_B_TO_A,
            gen_B_to_A, # Load into gen_B_to_A
            opt_gen,
            config.LEARNING_RATE,
        )
        load_checkpoint(
            config.CHECKPOINT_CRITIC_B,
            disc_B, # Load into disc_B
            opt_disc,
            config.LEARNING_RATE,
        )
        load_checkpoint(
            config.CHECKPOINT_CRITIC_A,
            disc_A, # Load into disc_A
            opt_disc,
            config.LEARNING_RATE,
        )

    # Updated dataset instantiation with SpleenDataset and generic root names
    # Now assumes TRAIN_DIR and VAL_DIR directly point to the image folders for Domain A and B
    dataset = SpleenDataset(
        root_domain_A=config.TRAIN_DIR,
        root_domain_B=config.VAL_DIR, # Using VAL_DIR for the second domain for training
        transform=config.transforms,
    )
    val_dataset = SpleenDataset(
        root_domain_A=config.VAL_DIR, # Assuming test images are also used for val_domain_A
        root_domain_B=config.VAL_DIR, # Assuming test images are also used for val_domain_B
        transform=config.transforms,
    )
    val_loader = DataLoader(
        val_dataset,
        batch_size=1,
        shuffle=False,
        pin_memory=True,
    )
    loader = DataLoader(
        dataset,
        batch_size=config.BATCH_SIZE,
        shuffle=True,
        num_workers=config.NUM_WORKERS,
        pin_memory=True,
    )
    g_scaler = torch.amp.GradScaler() # Updated to torch.amp.GradScaler
    d_scaler = torch.amp.GradScaler() # Updated to torch.amp.GradScaler

    for epoch in range(config.NUM_EPOCHS):
        train_fn(
            disc_A,
            disc_B,
            gen_B_to_A,
            gen_A_to_B,
            loader,
            opt_disc,
            opt_gen,
            L1,
            mse,
            d_scaler,
            g_scaler,
        )

        if config.SAVE_MODEL:
            save_checkpoint(gen_A_to_B, opt_gen, filename=config.CHECKPOINT_GEN_A_TO_B)
            save_checkpoint(gen_B_to_A, opt_gen, filename=config.CHECKPOINT_GEN_B_TO_A)
            save_checkpoint(disc_B, opt_disc, filename=config.CHECKPOINT_CRITIC_B)
            save_checkpoint(disc_A, opt_disc, filename=config.CHECKPOINT_CRITIC_A)


if __name__ == "__main__":
    main()


Overwriting train.py


In [ ]:
import importlib
import sys
import os

# Add current directory to Python path to ensure local modules are found
current_dir = os.getcwd()
if current_dir not in sys.path:
    sys.path.insert(0, current_dir)

print(f"DEBUG: Current working directory: {current_dir}")
print(f"DEBUG: sys.path after modification: {sys.path}")
utils_path = os.path.join(current_dir, 'utils.py')
print(f"DEBUG: Does '{utils_path}' exist? {os.path.exists(utils_path)}")
print(f"DEBUG: Content of current directory: {os.listdir(current_dir)}") # To be thorough

# Ensure config is reloaded first, as other modules depend on it
if 'config' in sys.modules:
    importlib.reload(sys.modules['config'])
else:
    import config # Import if not already loaded
print(f"DEBUG: config loaded successfully.")

# Reload utils, as it imports config
if 'utils' in sys.modules:
    importlib.reload(sys.modules['utils'])
else:
    import utils # This is the line that caused the error
print(f"DEBUG: utils loaded successfully.")

# Reload dataset, as it uses config and is part of the data loading pipeline
if 'dataset' in sys.modules:
    importlib.reload(sys.modules['dataset'])
else:
    import dataset
print(f"DEBUG: dataset loaded successfully.")

# Reload train, as it imports config, utils, and dataset
if 'train' in sys.modules:
    importlib.reload(sys.modules['train'])
else:
    import train
print(f"DEBUG: train loaded successfully.")

# Now run the main training function
print(f"DEBUG: Calling train.main()")
train.main()

DEBUG: Current working directory: /content
DEBUG: sys.path after modification: ['/content', '/env/python', '/usr/lib/python312.zip', '/usr/lib/python3.12', '/usr/lib/python3.12/lib-dynload', '', '/usr/local/lib/python3.12/dist-packages', '/usr/lib/python3/dist-packages', '/usr/local/lib/python3.12/dist-packages/IPython/extensions', '/root/.ipython']
DEBUG: Does '/content/utils.py' exist? True
DEBUG: Content of current directory: ['.config', 'train.py', 'config.py', 'discriminator_model.py', '__pycache__', 'utils.py', 'dataset.py', 'generator_model.py', 'drive', 'sample_data']
DEBUG: config loaded successfully.
DEBUG: utils loaded successfully.
DEBUG: dataset loaded successfully.
DEBUG: train loaded successfully.
DEBUG: Calling train.main()


100%|██████████| 3747/3747 [08:07<00:00,  7.68it/s, B_fake=0.12, B_real=0.865]


=> Saving checkpoint
=> Saving checkpoint
=> Saving checkpoint
=> Saving checkpoint


100%|██████████| 3747/3747 [07:03<00:00,  8.84it/s, B_fake=0.0269, B_real=0.96]


=> Saving checkpoint
=> Saving checkpoint
=> Saving checkpoint
=> Saving checkpoint


...

=> Saving checkpoint
=> Saving checkpoint
=> Saving checkpoint
=> Saving checkpoint


100%|██████████| 3747/3747 [07:08<00:00,  8.74it/s, B_fake=0.0157, B_real=0.97]


=> Saving checkpoint
=> Saving checkpoint
=> Saving checkpoint
=> Saving checkpoint


100%|██████████| 3747/3747 [07:05<00:00,  8.80it/s, B_fake=0.0132, B_real=0.973]


=> Saving checkpoint
=> Saving checkpoint
=> Saving checkpoint
=> Saving checkpoint


In [ ]:
import os
import config

# List contents of the TRAIN_DIR
print(f"Contents of {config.TRAIN_DIR}:")
!ls -F "{config.TRAIN_DIR}"

# Also check the VAL_DIR
print(f"\nContents of {config.VAL_DIR}:")
!ls -F "{config.VAL_DIR}"

Contents of /content/drive/MyDrive/Processed Datasets/US.v7i.yolov8-obb Roboflow/train/images:
10113_jpg.rf.27bb751e6d7c673ab41776fb29e09018.jpg
10113_jpg.rf.8b9feda5141f1011b4d8acc46ce8e97b.jpg
10113_jpg.rf.f7c809c2896573fefb659d5055feb939.jpg
10114_jpg.rf.1e63b74443c1f519a568b98a018dd92a.jpg
10114_jpg.rf.b7397fc94c485dee5b5568968261b97a.jpg
10114_jpg.rf.ec5ebd7a567ee5d33756abc77ee50211.jpg
10115_jpg.rf.2adf764d9db5d0756dcc75cc1b7dd37a.jpg
10115_jpg.rf.3a309c3884698b041de594cb90e8f99c.jpg
10115_jpg.rf.6b618d15906dcdbfc5b4e746621d166a.jpg
10116_jpg.rf.4d886cbaf63ef9f8f432e4bf6a203985.jpg
10116_jpg.rf.54e241067ebea7a712b4b019b02e533f.jpg
10116_jpg.rf.c48e9e4eb475437e20ee16745b35a357.jpg
10117_jpg.rf.e68b2452625c543f555f47a4f1686ae7.jpg
10117_jpg.rf.f71fcfa7a3a40c65464b1ed543c0d0ef.jpg
10117_jpg.rf.fb8a80933583de114f8870fe6cf77fce.jpg
10118_jpg.rf.0101dfefbc8b5b72f065901e525a1402.jpg
10118_jpg.rf.780aea24492a49b88ee83d9796715d06.jpg
10118_jpg.rf.e0e12aadca531bd2528a159e30451d4d.jpg
10119

In [ ]:
#Downloads saved images
import zipfile
import os
from google.colab import files

# Define the directory to zip
image_dir = 'saved_images'
zip_filename = 'saved_images.zip'

# Create a ZipFile object
with zipfile.ZipFile(zip_filename, 'w') as zipf:
    # Iterate over all the files in the directory
    for root, dirs, files_in_dir in os.walk(image_dir):
        for file in files_in_dir:
            file_path = os.path.join(root, file)
            # Add file to zip, preserving directory structure within the zip
            zipf.write(file_path, os.path.relpath(file_path, image_dir))

print(f"'{image_dir}' directory has been compressed into '{zip_filename}'")

# Download the zip file to your local machine
files.download(zip_filename)

'saved_images' directory has been compressed into 'saved_images.zip'


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>